In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data=pd.read_csv('https://raw.githubusercontent.com/priyamnagar/feature_selection_titanic/master/titanic3.csv')
y=data['survived']
X=data.copy()
del X['survived']

In [ ]:
data.head(50)

In [ ]:
data.describe()

**Missing** Values

In [ ]:
data.isnull().sum().sort_values(ascending=True)

In [ ]:
#body
X['body_null']=np.where(X.body.isnull(),1,0)
X.body.fillna(0,inplace=True)

In [ ]:
#cabin
temp_cabin=X.cabin.str.split(expand=True)[0]
X['cabin_char']=temp_cabin.str[0]
#X['cabin_num']=temp_cabin.str[1:]
del X['cabin']

X['cabin_char']=np.where(X.cabin_char.isnull(),'M',X.cabin_char)

In [ ]:
X['cabin_char'].value_counts()

In [ ]:
X['cabin_char']=np.where(X.cabin_char.isnull(),'M', X.cabin_char)

In [ ]:
#boat
temp_boat=X.boat.str.split(expand=True).rename(columns={0:'boat_1',1:'boat_2',2:'boat_3'})
X['boat_char']=np.where(temp_boat.boat_1.str.isdigit(),np.nan,temp_boat.boat_1)
#X['boat_num']=np.where(temp_boat.boat_1.str.isdigit(),temp_boat.boat_1,np.nan)

X['boat_char']=np.where(X.boat_char.isnull(),'M',X.boat_char)
#X['boat_num'].fillna(0,inplace=True)
del X['boat']

In [ ]:
X['boat_char'].value_counts()

In [ ]:
#home.dest
del X['home.dest']

#age
X.age.fillna(X.age.mean(),inplace=True)

#Embarked
X['embarked']=np.where(X.embarked.isnull(),'M',X.embarked)

#fare
X.fare.fillna(X.fare.median(),inplace=True)

In [ ]:
#LabelEncoding
#name
X['name']=X.name.str.split(',',expand=True)[1].str.split('.',expand=True)[0]
X['name']=X['name'].map({' Mrs': 0,  ' Mr': 1, ' Miss': 2, ' Master': 3, ' Col': 1, ' Mme': 0, ' Dr': 4, ' Major': 1, ' Capt': 1, ' Lady': 0, ' Sir': 1, ' Mlle': 0, ' Dona': 0, ' Jonkheer': 1, ' the Countess': 0, ' Don': 1, ' Rev': 1, ' Ms': 0})

#sex
X['sex']=X['sex'].map({'male':0,'female':1})

del X['ticket']

#Embarked
X['embarked']=X.embarked.map({'M':0,'S':1,'C':2,'Q':3})


#Cabin_char
X['cabin_char']=X.cabin_char.map({k:i for i,k in enumerate(X.cabin_char.unique())})

#Boat_char
X['boat_char']=X.boat_char.map({k:i for i,k in enumerate(X.boat_char.unique())})


In [ ]:
#One hot encoding

#name
temp_name=pd.get_dummies(X.name,drop_first=True)
X=pd.concat([X,temp_name],axis=1)
del X['name']

#embarked
temp_embarked=pd.get_dummies(X.embarked,drop_first=True).rename(columns={1:5,2:6,3:7})
X=pd.concat([X,temp_embarked],axis=1)
del X['embarked']


In [ ]:
X

In [ ]:
## train test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=50)

In [ ]:
from xgboost import XGBClassifier
model_filter=XGBClassifier()
model_filter.fit(X_train,y_train)

y_pred=model_filter.predict(X_test)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test,y_pred)

print (classification_report(y_test, y_pred))

cm = confusion_matrix( y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

# Feature Selection

## Filter Methods

#### Baixa Variância

In [ ]:

X_train_filter=X_train.copy()
y_train_filter=y_train.copy()
X_test_filter=X_test.copy()
y_test_filter=y_test.copy()

X_train_filter= X_train_filter.rename(str,axis="columns")
X_test_filter= X_test_filter.rename(str,axis="columns")

from sklearn.feature_selection import VarianceThreshold
sel=VarianceThreshold(threshold=0.01)
sel.fit(X_train_filter)
sel.transform(X_test_filter)

print(X_train_filter.columns[sel.get_support()])

del X_train_filter['4']
del X_test_filter['4']

from xgboost import XGBClassifier
model_filter=XGBClassifier()
model_filter.fit(X_train_filter,y_train_filter)

y_pred_filter=model_filter.predict(X_test_filter)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_filter,y_pred_filter)

print (classification_report(y_test, y_pred_filter))

cm = confusion_matrix( y_test, y_pred_filter)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

#### Método de correlação (variável alvo)

In [ ]:
X_train_filter=X_train.copy()
y_train_filter=y_train.copy()
X_test_filter=X_test.copy()
y_test_filter=y_test.copy()

corrmat={}
for i in X_train_filter.columns.values:
    corrmat[i]=X_train_filter[i].corr(y_train_filter)

print(corrmat)

In [ ]:
#Deleta correlação < 10%

del X_train_filter[7]
del X_train_filter[4]
del X_train_filter[3]
del X_train_filter['parch']
del X_train_filter['sibsp']
del X_train_filter['age']
del X_test_filter[7]
del X_test_filter[4]
del X_test_filter[3]
del X_test_filter['parch']
del X_test_filter['sibsp']
del X_test_filter['age']

from xgboost import XGBClassifier
model_filter=XGBClassifier()
model_filter.fit(X_train_filter,y_train_filter)


y_pred_filter=model_filter.predict(X_test_filter)

from sklearn.metrics import confusion_matrix
metric_filter=confusion_matrix(y_test_filter,y_pred_filter)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_filter,y_pred_filter)

print (classification_report(y_test, y_pred_filter))

cm = confusion_matrix( y_test, y_pred_filter)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

## Embedded methods

In [ ]:
X_train_embedded=X_train.copy()
y_train_embedded=y_train.copy()
X_test_embedded=X_test.copy()
y_test_embedded=y_test.copy()

from xgboost import XGBClassifier
xgb=XGBClassifier()
xgb.fit(X_train_embedded,y_train_embedded)

feat_importance = xgb.feature_importances_ * 100

importance_dict = {}
for i, col in enumerate(X_train_embedded):
    importance_dict[col] = feat_importance[i]

print(importance_dict)


for i, (col, importance) in enumerate(importance_dict.items()):
    if importance < 1:
        X_train_embedded.drop(col, axis=1, inplace=True)
        X_test_embedded.drop(col, axis=1, inplace=True)
        print(f'del: {col}')

xgb=XGBClassifier()
xgb.fit(X_train_embedded,y_train_embedded)
y_pred_embedded=xgb.predict(X_test_embedded)

from sklearn.metrics import confusion_matrix
metric_embedded=confusion_matrix(y_test_embedded,y_pred_embedded)

accuracy_embedded=(metric_embedded[0][0]+metric_embedded[1][1])/sum(sum(metric_embedded))*100
print('Accuracy using embedded : ', accuracy_embedded)

## Wrapper methods

#### Forward feature selection

In [ ]:

X_train_wrapper=X_train.copy()
y_train_wrapper=y_train.copy()
X_test_wrapper=X_test.copy()
y_test_wrapper=y_test.copy()

from xgboost import XGBClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
sfs1=SFS(XGBClassifier(n_jobs=4),k_features=9,forward=True,floating=False,verbose=2,scoring='roc_auc',cv=3)
sfs1.fit(np.array(X_train_wrapper),y_train_wrapper)

In [ ]:
x_train_forward=sfs1.transform(X_train_wrapper)
x_test_forward=sfs1.transform(X_test_wrapper)

from xgboost import XGBClassifier
model_forward=XGBClassifier()
model_forward.fit(x_train_forward,y_train_wrapper)

y_pred_forward=model_forward.predict(x_test_forward)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_wrapper,y_pred_forward)
print (classification_report(y_test_wrapper, y_pred_forward))

cm = confusion_matrix( y_test_wrapper, y_pred_forward)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

#### Backward feature selection

In [ ]:
X_train_wrapper=X_train.copy()
y_train_wrapper=y_train.copy()
X_test_wrapper=X_test.copy()
y_test_wrapper=y_test.copy()
from xgboost import XGBClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
sfs2=SFS(XGBClassifier(n_jobs=4),k_features=5,forward=False,floating=False,verbose=2,scoring='roc_auc',cv=3)
sfs2.fit(np.array(X_train_wrapper),y_train_wrapper)

In [ ]:
x_train_backward=sfs2.transform(X_train_wrapper)
x_test_backward=sfs2.transform(X_test_wrapper)

from xgboost import XGBClassifier
model_backward=XGBClassifier()
model_backward.fit(x_train_backward,y_train_wrapper)

y_pred_backward=model_backward.predict(x_test_backward)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_wrapper,y_pred_backward)
print (classification_report(y_test_wrapper, y_pred_backward))

cm = confusion_matrix( y_test_wrapper, y_pred_backward)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

#### Exhaustive feature selection


In [ ]:
X_train_wrapper=X_train.copy()
y_train_wrapper=y_train.copy()
X_test_wrapper=X_test.copy()
y_test_wrapper=y_test.copy()

from xgboost import XGBClassifier
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS
sfs3=EFS(XGBClassifier(n_jobs=4),min_features=1,max_features=3,scoring='roc_auc',print_progress=True,cv=2)
sfs3.fit(np.array(X_train_wrapper),y_train_wrapper)

## Statistical Filter Methods

#### Ganho de Informação (Informação mútua)

In [ ]:
X_train_filter=X_train.copy()
y_train_filter=y_train.copy()
X_test_filter=X_test.copy()
y_test_filter=y_test.copy()

X_train_filter= X_train_filter.rename(str,axis="columns")
X_test_filter= X_test_filter.rename(str,axis="columns")

from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.feature_selection import SelectKBest,SelectPercentile
#mi=mutual_info_classif(X_train_filter,y_train_filter)
select=SelectKBest(mutual_info_classif,k=12).fit(X_train_filter,y_train_filter)
X_train_filter=X_train_filter[X_train_filter.columns[select.get_support()].values]
X_test_filter=X_test_filter[X_test_filter.columns[select.get_support()].values]

from xgboost import XGBClassifier
model_filter=XGBClassifier()
model_filter.fit(X_train_filter,y_train_filter)


y_pred_filter=model_filter.predict(X_test_filter)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_filter,y_pred_filter)

print (classification_report(y_test, y_pred_filter))

cm = confusion_matrix( y_test, y_pred_filter)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

#### Score de Fisher

In [ ]:
X_train_filter=X_train.copy()
y_train_filter=y_train.copy()
X_test_filter=X_test.copy()
y_test_filter=y_test.copy()

X_train_filter= X_train_filter.rename(str,axis="columns")
X_test_filter= X_test_filter.rename(str,axis="columns")

from sklearn.feature_selection import SelectKBest, chi2
select=SelectKBest(chi2,k=10).fit(X_train_filter,y_train_filter)
X_train_filter=X_train_filter[X_train_filter.columns[select.get_support()].values]
X_test_filter=X_test_filter[X_test_filter.columns[select.get_support()].values]
from xgboost import XGBClassifier
model_filter=XGBClassifier()
model_filter.fit(X_train_filter,y_train_filter)


y_pred_filter=model_filter.predict(X_test_filter)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_filter,y_pred_filter)

print (classification_report(y_test, y_pred_filter))

cm = confusion_matrix( y_test, y_pred_filter)
disp = ConfusionMatrixDisplay(cm)
disp.plot()

#### Univariado (Anova)

In [ ]:
X_train_filter=X_train.copy()
y_train_filter=y_train.copy()
X_test_filter=X_test.copy()
y_test_filter=y_test.copy()

X_train_filter= X_train_filter.rename(str,axis="columns")
X_test_filter= X_test_filter.rename(str,axis="columns")

from sklearn.feature_selection import SelectKBest, f_classif
select=SelectKBest(f_classif,k=11).fit(X_train_filter,y_train_filter)
X_train_filter=X_train_filter[X_train_filter.columns[select.get_support()].values]
X_test_filter=X_test_filter[X_test_filter.columns[select.get_support()].values]

from xgboost import XGBClassifier
model_filter=XGBClassifier()
model_filter.fit(X_train_filter,y_train_filter)


y_pred_filter=model_filter.predict(X_test_filter)

from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, confusion_matrix
metric_filter=confusion_matrix(y_test_filter,y_pred_filter)

print (classification_report(y_test, y_pred_filter))

cm = confusion_matrix( y_test, y_pred_filter)
disp = ConfusionMatrixDisplay(cm)
disp.plot()